# The last few commands do connect and appear to get a response from "Identify"

In [14]:
import socket
import time

In [15]:
AUTOSAMPLER_IP = '192.168.1.101'
AUTOSAMPLER_PORT = 50185

def identify_instrument(ip, port):
    identify_xml = """
    <Gilson>
        <CommandData>
            <Commands>
                <Command>
                    <CommandType>Local</CommandType>
                    <CommandInfo>
                        <InstrumentInfo>
                            <DeviceName>GX-27x</DeviceName>
                            <DeviceId>35</DeviceId>
                        </InstrumentInfo>
                        <SequenceNumber>1</SequenceNumber>
                        <Synchronize>True</Synchronize>
                        <Selected>False</Selected>
                        <CommandXML>
                            <CommandName>Identify</CommandName>
                        </CommandXML>
                    </CommandInfo>
                </Command>
            </Commands>
        </CommandData>
    </Gilson>
    """.strip()

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(5)
        try:
            print(f"Connecting to {ip}:{port}")
            s.connect((ip, port))
            s.sendall(identify_xml.encode('ascii'))
            response = s.recv(8192).decode('ascii')
            print("Response received:\n", response)
            return response
        except Exception as e:
            print("Error:", e)
            return None

# Run Identify command
identify_instrument(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)


Connecting to 192.168.1.101:50185
Response received:
 


''

In [16]:
def identify_instrument_multi_block(ip, port):
    message_id = 2000  # Arbitrary ID, increment for each command
    gilson_xml_block0 = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
    <CommandData>
        <Commands>
            <Command>
                <CommandType>Local</CommandType>
                <CommandInfo>
                    <InstrumentInfo>
                        <DeviceName>GX-27x</DeviceName>
                        <DeviceId>35</DeviceId>
                    </InstrumentInfo>
                    <SequenceNumber>1</SequenceNumber>
                    <Synchronize>True</Synchronize>
                    <Selected>False</Selected>
                    <CommandXML>
                        <CommandName>Identify</CommandName>
                    </CommandXML>
                </CommandInfo>
            </Command>
        </Commands>
    </CommandData>
</Gilson>
</Message>
""".strip()

    gilson_xml_block1 = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>
""".strip()

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(5)
        try:
            print(f"Connecting to {ip}:{port}")
            s.connect((ip, port))

            # Send both blocks
            s.sendall(gilson_xml_block0.encode('ascii'))
            s.sendall(gilson_xml_block1.encode('ascii'))

            response = s.recv(8192).decode('ascii')
            print("Response received:\n", response)
            return response
        except Exception as e:
            print("Error:", e)
            return None


In [17]:
identify_instrument_multi_block(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)

Connecting to 192.168.1.101:50185
Response received:
 


''

In [25]:
def identify_over_persistent_socket(ip, port):
    message_id = 2001

    # Block 0 of 2
    block_0 = f"""<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
    <CommandData>
        <Commands>
            <Command>
                <CommandType>Local</CommandType>
                <CommandInfo>
                    <InstrumentInfo>
                        <DeviceName>GX-27x</DeviceName>
                        <DeviceId>35</DeviceId>
                    </InstrumentInfo>
                    <SequenceNumber>1</SequenceNumber>
                    <Synchronize>True</Synchronize>
                    <Selected>False</Selected>
                    <CommandXML>
                        <CommandName>Identify</CommandName>
                    </CommandXML>
                </CommandInfo>
            </Command>
        </Commands>
    </CommandData>
</Gilson>
</Message>
"""

    # Block 1 of 2 (empty but required)
    block_1 = f"""<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>"""

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        print(f"Connecting to {ip}:{port}...")
        s.connect((ip, port))

        # Send both blocks down same socket
        print("Sending Block 0...")
        s.sendall(block_0.encode('ascii'))
        time.sleep(0.1)
        print("Sending Block 1...")
        s.sendall(block_1.encode('ascii'))

        # Wait longer for response
        time.sleep(0.5)
        print("Receiving...")
        response = s.recv(8192).decode('ascii')
        print("Response:\n", response)

        s.close()
        return response
    except Exception as e:
        print("Error:", e)
        return None


In [26]:
identify_over_persistent_socket(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)

Connecting to 192.168.1.101:50185...
Sending Block 0...
Sending Block 1...
Receiving...
Error: [WinError 10053] An established connection was aborted by the software in your host machine


In [27]:
import socket
import time

def identify_over_persistent_socket(ip, port):
    message_id = 2002

    # Block 0
    block_0 = f"""<Message MessageId="{message_id}" BlockCount="2" BlockId="0">\r\n<Gilson>
<CommandData>
    <Commands>
        <Command>
            <CommandType>Local</CommandType>
            <CommandInfo>
                <InstrumentInfo>
                    <DeviceName>GX-27x</DeviceName>
                    <DeviceId>35</DeviceId>
                </InstrumentInfo>
                <SequenceNumber>1</SequenceNumber>
                <Synchronize>True</Synchronize>
                <Selected>False</Selected>
                <CommandXML>
                    <CommandName>Identify</CommandName>
                </CommandXML>
            </CommandInfo>
        </Command>
    </Commands>
</CommandData>
</Gilson>
</Message>\r\n"""

    # Block 1 (empty body but still required)
    block_1 = f"""<Message MessageId="{message_id}" BlockCount="2" BlockId="1">\r\n</Message>\r\n"""

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        print(f"Connecting to {ip}:{port}...")
        s.connect((ip, port))

        print("Sending Block 0...")
        s.sendall(block_0.encode('ascii'))

        time.sleep(0.1)  # slight wait before block 1
        print("Sending Block 1...")
        s.sendall(block_1.encode('ascii'))

        time.sleep(0.5)  # let the device process
        print("Receiving...")
        response = s.recv(8192).decode('ascii')
        print("Response:\n", response)

        s.close()
        return response
    except Exception as e:
        print("Error:", e)
        return None


In [28]:
identify_over_persistent_socket(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)

Connecting to 192.168.1.101:50185...
Sending Block 0...
Sending Block 1...
Receiving...
Error: [WinError 10053] An established connection was aborted by the software in your host machine


In [29]:
def identify_single_send(ip, port):
    message_id = 2003

    # Full message includes both blocks in one send
    full_message = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>GX-27x</DeviceName>
            <DeviceId>35</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>40</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Identify</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>
</Message>
<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>
""".replace('\n', '\r\n').strip()

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        print(f"Connecting to {ip}:{port}...")
        s.connect((ip, port))

        print("Sending full message (blocks 0 & 1 together)...")
        s.sendall(full_message.encode('ascii'))

        time.sleep(1)
        print("Receiving...")
        response = s.recv(8192).decode('ascii')
        print("Response:\n", response)

        s.close()
        return response
    except Exception as e:
        print("Error:", e)
        return None


In [32]:
identify_single_send(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)

Connecting to 192.168.1.101:50185...
Sending full message (blocks 0 & 1 together)...
Receiving...
Response:
 


''

In [33]:
def send_admin_command(ip, port):
    message_id = 2004
    full_message = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>System</DeviceName>
            <DeviceId>-1</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>0</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Admin</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>
</Message>
<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>
""".replace('\n', '\r\n').strip()

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        print(f"Connecting to {ip}:{port}...")
        s.connect((ip, port))

        print("Sending Admin command...")
        s.sendall(full_message.encode('ascii'))

        time.sleep(1)
        print("Receiving...")
        response = s.recv(8192).decode('ascii')
        print("Response:\n", response)

        s.close()
        return response
    except Exception as e:
        print("Error:", e)
        return None


In [34]:
send_admin_command(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)

Connecting to 192.168.1.101:50185...
Sending Admin command...
Receiving...
Response:
 


''

In [35]:
def request_assigned_port(ip, admin_port):
    message = """
<Message MessageId="1984" BlockCount="1" BlockId="0">
<Gilson>
    <GilsonConnect>Gilson Ethernet Utility</GilsonConnect>
</Gilson>
</Message>
""".replace('\n', '\r\n').strip()

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        print(f"Connecting to admin port {ip}:{admin_port}...")
        s.connect((ip, admin_port))
        s.sendall(message.encode('ascii'))

        time.sleep(1)
        response = s.recv(4096).decode('ascii')
        print("Response:\n", response)

        s.close()

        # Try to extract the port
        if "<ParameterName>ResponseValue</ParameterName>" in response:
            port_value = response.split("<ParameterValue>")[2].split("</ParameterValue>")[0]
            return int(port_value)
        else:
            return None
    except Exception as e:
        print("Error:", e)
        return None


In [36]:
request_assigned_port(AUTOSAMPLER_IP, AUTOSAMPLER_PORT)

Connecting to admin port 192.168.1.101:50185...
Response:
 <Message MessageId="2008" BlockCount="1" BlockId="0"><Gilson><Response><CommandName>Connection Request</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue>50190</ParameterValue></Parameter></Parameters></Response></Gilson></Message>


50190

In [37]:
assigned_port = request_assigned_port('192.168.1.101', 50185)

Connecting to admin port 192.168.1.101:50185...
Response:
 <Message MessageId="2009" BlockCount="1" BlockId="0"><Gilson><Response><CommandName>Connection Request</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue>50191</ParameterValue></Parameter></Parameters></Response></Gilson></Message>


In [38]:
def send_admin_command(ip, port):
    message_id = 2005
    full_message = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>System</DeviceName>
            <DeviceId>-1</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>0</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Admin</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>
</Message>
<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>
""".replace('\n', '\r\n').strip()

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        s.connect((ip, port))
        s.sendall(full_message.encode('ascii'))
        time.sleep(1)
        response = s.recv(8192).decode('ascii')
        print("Admin Response:\n", response)
        s.close()
    except Exception as e:
        print("Admin Error:", e)


In [39]:
def send_identify_command(ip, port):
    message_id = 2006
    full_message = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>GX-27x</DeviceName>
            <DeviceId>35</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>1</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Identify</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>
</Message>
<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>
""".replace('\n', '\r\n').strip()

    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(5)
        s.connect((ip, port))
        s.sendall(full_message.encode('ascii'))
        time.sleep(1)
        response = s.recv(8192).decode('ascii')
        print("Identify Response:\n", response)
        s.close()
    except Exception as e:
        print("Identify Error:", e)   

In [40]:
assigned_port = request_assigned_port('192.168.1.101', 50185)
if assigned_port:
    send_admin_command('192.168.1.101', assigned_port)
    send_identify_command('192.168.1.101', assigned_port)


Connecting to admin port 192.168.1.101:50185...
Response:
 <Message MessageId="2010" BlockCount="1" BlockId="0"><Gilson><Response><CommandName>Connection Request</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue>50190</ParameterValue></Parameter></Parameters></Response></Gilson></Message>
Admin Response:
 <Message MessageId="2011" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><